# Phase 3: binary baseline
Compare Logistic Regression and Random Forest using recorded validation results, then inspect the selected model’s recorded test result. This notebook does not retrain or rescore the test set. Run `python -m src.train` first if results are missing. Use the NetworkMonitoring (Python 3.12) kernel.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display, Image
from src.predict import predict_frame
path = ROOT / "reports/binary_baseline_metrics.json"
if not path.exists():
    raise RuntimeError("Run python -m src.train from the project root first")
results = json.loads(path.read_text())

## 1. Experiment design
BENIGN=0, ATTACK=1. Models fit training only. Validation macro F1 selects the model, with lower false-positive rate breaking ties. Balanced class weights and a fixed 0.5 decision threshold are used. Only the winner receives a final test score. Class-capped sampling changes prevalence; this is not a deployment evaluation.

In [ ]:
print("Split sizes:", results["split_records"])
print("Selection rule:", results["selection_rule"])
print("Selected:", results["selected_model"])
print("Source revision:", results["dataset_revision"])

## 2. Validation comparison
Macro F1 gives benign and attack classes equal weight. Benign false-positive rate measures the share of normal flows incorrectly flagged.

In [ ]:
metrics = ["accuracy", "precision_attack", "recall_attack", "f1_attack", "macro_f1", "false_positive_rate"]
comparison = pd.DataFrame({name: item["metrics"] for name, item in results["candidates"].items()}).T
comparison.loc["majority_dummy"] = pd.Series(results["dummy"]["metrics"])
display(comparison[metrics].astype(float))
comparison[["macro_f1", "false_positive_rate"]].astype(float).plot.bar(figsize=(9, 4))
plt.ylim(0, 1.05)
plt.title("Validation metrics only")
plt.tight_layout()
plt.show()
for name, item in results["candidates"].items():
    print(name, "converged:", item["converged"], "warnings:", item["warnings"])

## 3. Selected-model test result
This is the recorded final baseline measurement. Do not use this score to choose new thresholds or hyperparameters.

In [ ]:
display(pd.Series({k: results["test"][k] for k in metrics}, name="test"))
display(pd.DataFrame(results["test"]["classification_report"]).T)
display(Image(filename=str(ROOT / "reports/binary_test_confusion_matrix.png")))

## 4. Offline inference with the saved pipeline
Use validation examples for this demonstration. Labels and source metadata are excluded automatically. Attack scores are uncalibrated; they are not guaranteed probabilities on real traffic.

In [ ]:
artifact = joblib.load(ROOT / "models/binary_detector.joblib")
examples = pd.read_parquet(ROOT / "data/processed/phase2/validation_raw.parquet").head(10)
predictions = predict_frame(examples, artifact)
display(pd.concat([examples[["Label"]], predictions], axis=1))

## 5. Report and next steps
The binary baseline is complete. Multiclass classification and live/replayed traffic validation remain separate next steps. Rare attacks and residual session/time similarity limit conclusions.

In [ ]:
display(Markdown((ROOT / "reports/phase3_binary_baseline.md").read_text()))